In [1]:
import polars as pl
import polars.selectors as cs
import wandb
import numpy as np
import plotly.express as px
import matplotlib.pyplot as plt

api = wandb.Api()
artifact = api.artifact("flood-forecasting/flood-dataset:latest")
artifact_dir = artifact.download()

df = (
    pl.scan_parquet(f"{artifact_dir}/flood_model.parquet")
    .limit(100_000)
    .filter(pl.col("observation_hour") < pl.lit("2024-01-01").str.to_datetime())
    .collect()
)

print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from C:\Users\sacha\_netrc.
wandb: Downloading large artifact 'flood-dataset:latest', 4549.24MB. 1 files...
wandb:   1 of 1 files downloaded.  
Done. 00:00:00.4 (10780.2MB/s)


Loaded: 91,039 rows x 460 columns


In [2]:
numeric_df = df.select(cs.numeric()).to_pandas()

target_corr = (
    numeric_df.corr()[["streamflow_cfs_mean"]]
    .drop(index="streamflow_cfs_mean")
    .sort_values("streamflow_cfs_mean", ascending=False)
)

low_corr = target_corr[target_corr["streamflow_cfs_mean"].abs() < 0.1]

print(f"Total features: {len(target_corr)}")
print(f"Low correlation with target (<0.1): {len(low_corr)}")
print("\nTop 15 correlated:")
print(target_corr.head(15))
print("\nBottom 15 correlated:")
print(target_corr.tail(15))

Total features: 446
Low correlation with target (<0.1): 143

Top 15 correlated:
                              streamflow_cfs_mean
streamflow_cfs_max                       0.999985
streamflow_cfs_min                       0.999984
streamflow_cfs_target_1h                 0.999939
rip800_12                                0.496638
snow_ice_nlcd06                          0.496621
cdl_durum_wheat                          0.495129
padcat1_pct_basin                        0.477317
rip100_12                                0.473858
artificial_path_mainstem_pct             0.466895
wet_pc_ug2                               0.452502
padcat2_pct_basin                        0.445232
barren_nlcd06                            0.404642
wet_pc_ug1                               0.403148
topwet                                   0.402845
hga                                      0.389804

Bottom 15 correlated:
                       streamflow_cfs_mean
cdl_wwht_soy_dbl_crop                  NaN
cdl_pasture

In [3]:
# drop NaN correlation or absolute correlation < 0.1
keep_always = [
    "streamflow_cfs_mean", "streamflow_cfs_max", "streamflow_cfs_min",
    "DRAIN_SQKM",
    "longitude",
    "latitude"
]

drop_cols = target_corr[
    (target_corr["streamflow_cfs_mean"].abs() < 0.2) |
    (target_corr["streamflow_cfs_mean"].isna())
].index.tolist()

force_drop = [
    "pnv_pc_u10",       
    "mains800_52",      
    "rip800_12",        
    "streamflow_cfs_target_1h", 
    "gage_height_ft_target_1h"

]

drop_cols = [c for c in drop_cols if c not in keep_always]
drop_cols += [c for c in force_drop if c in numeric_df.columns and c not in keep_always]

drop_cols = [c for c in drop_cols if c not in keep_always]

reduced_df = numeric_df.drop(columns=drop_cols)

print(f"Dropped: {len(drop_cols)} columns")
print(f"Remaining: {reduced_df.shape[1]} columns")

Dropped: 337 columns
Remaining: 111 columns


In [4]:
corr_reduced = reduced_df.corr()

fig = px.imshow(
    corr_reduced,
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    title="Correlation Matrix: High-Signal Features (|r| > 0.2 with target)",
    width=1200, height=1200
)

fig.update_layout(coloraxis_colorbar=dict(title="r"))
fig.show()

In [5]:
corr_matrix = reduced_df.corr().abs()

# Find pairs with r > 0.9 
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_pairs = (
    upper.stack()
    .reset_index()
    .rename(columns={"level_0": "feature_1", "level_1": "feature_2", 0: "r"})
    .query("r > 0.9")
    .sort_values("r", ascending=False)
)

print(f"Hyper-correlated pairs (r > 0.9): {len(high_corr_pairs)}")
print(high_corr_pairs.to_string())

Hyper-correlated pairs (r > 0.9): 837
                         feature_1                     feature_2         r
4674                    tbi_cl_smj                    tec_cl_smj  1.000000
114                      longitude                   longitude_1  1.000000
2705                    tmp_dc_smx                    tmp_dc_s07  1.000000
219            streamflow_cfs_mean            streamflow_cfs_max  0.999985
220            streamflow_cfs_mean            streamflow_cfs_min  0.999984
749                     DRAIN_SQKM            nwis_drainage_area  0.999973
327             streamflow_cfs_max            streamflow_cfs_min  0.999943
1681                        perdun                    tec_cl_smj  0.999906
1680                        perdun                    tbi_cl_smj  0.999906
2865                    tmp_dc_s05                    tmp_dc_s06  0.999341
3407                    aet_mm_syr                    aet_mm_s07  0.999148
2627                    tmp_dc_syr                    tmp_dc_s

In [6]:
from collections import Counter

# Count how many times each feature appears in a pair
all_features = high_corr_pairs["feature_1"].tolist() + high_corr_pairs["feature_2"].tolist()
freq = Counter(all_features)

to_drop = set()
for _, row in high_corr_pairs.iterrows():
    f1, f2 = row["feature_1"], row["feature_2"]
    if f1 in to_drop or f2 in to_drop:
        continue
    # Drop whichever appears in more pairs 
    drop = f1 if freq[f1] >= freq[f2] else f2
    if drop not in keep_always:
        to_drop.add(drop)

final_df = reduced_df.drop(columns=list(to_drop))

print(f"Dropped: {len(to_drop)} hyper-correlated columns")
print(f"Final feature count: {final_df.shape[1]}")
print(f"Kept: {list(final_df.columns)}")

Dropped: 93 hyper-correlated columns
Final feature count: 18
Kept: ['latitude', 'longitude', 'streamflow_cfs_mean', 'streamflow_cfs_max', 'streamflow_cfs_min', 'specific_humidity_kgkg', 'DRAIN_SQKM', 'artificial_path_pct', 'wb5100_ann_mm', 'snw_pc_syr', 'snow_ice_nlcd06', 'barren_nlcd06', 'mains100_plant', 'hga', 'hgc', 'bulk_density_avg', 'elev_max_m', 'aspect_deg']


In [7]:
# Correlation with target
final_target_corr = (
    final_df.corr()[["streamflow_cfs_mean"]]
    .drop(index="streamflow_cfs_mean")  
    .sort_values("streamflow_cfs_mean", ascending=False)
    .round(3)
)
print("Correlation with streamflow_cfs_mean:")
print(final_target_corr.to_string())

Correlation with streamflow_cfs_mean:
                        streamflow_cfs_mean
streamflow_cfs_max                    1.000
streamflow_cfs_min                    1.000
snow_ice_nlcd06                       0.497
barren_nlcd06                         0.405
hga                                   0.390
hgc                                   0.347
bulk_density_avg                      0.316
artificial_path_pct                   0.292
specific_humidity_kgkg                0.237
snw_pc_syr                            0.226
elev_max_m                            0.212
DRAIN_SQKM                            0.206
wb5100_ann_mm                         0.204
latitude                             -0.009
aspect_deg                           -0.214
mains100_plant                       -0.270
longitude                            -0.410


In [8]:
corr_final = final_df.corr()

fig = px.imshow(
    corr_final,
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    title=f"Final Feature Correlation Matrix ({len(final_df.columns)} features)",
    width=900, height=900,
    text_auto=".2f" 
)

fig.update_layout(coloraxis_colorbar=dict(title="r"))
fig.show()

## Selected Features

Starting from 460 columns, we reduced to 17 features using correlation analysis on ~91k rows (2007–2023).

**Process:** dropped features with |r| < 0.2 against streamflow, then dropped hyper-correlated pairs (|r| > 0.9) using a greedy algorithm. `DRAIN_SQKM` and `longitude` were force-kept due to physical importance.

**Dynamic inputs:** `streamflow_cfs_mean`, `streamflow_cfs_max`, `streamflow_cfs_min`

**Static attributes:**

| Feature | Category | Description | r |
|---|---|---|---|
| `snow_ice_nlcd06` | Snow | Snow/ice land cover % | +0.50 |
| `snw_pc_syr` | Snow | Annual snow cover % | +0.23 |
| `longitude` | Spatial | East-west position, proxy for aridity | -0.41 |
| `latitude` | Spatial | North-south position | -0.009 |
| `hga` | Soils | Soil group A % (high infiltration) | +0.39 |
| `hgc` | Soils | Soil group C % (high runoff) | +0.35 |
| `bulk_density_avg` | Soils | Soil compactness, affects infiltration | +0.32 |
| `barren_nlcd06` | Land Cover | Bare land %, high runoff areas | +0.41 |
| `mains100_plant` | Land Cover | Vegetation % along mainstem, dampens peaks | -0.27 |
| `artificial_path_pct` | Hydrology | % artificial stream network (canals etc.) | +0.29 |
| `wb5100_ann_mm` | Hydrology | Annual water balance (precip minus ET) | +0.20 |
| `specific_humidity_kgkg` | Climate | Atmospheric moisture | +0.24 |
| `DRAIN_SQKM` | Basin | Drainage area (sq km) | +0.21 |
| `elev_max_m` | Topography | Max basin elevation, controls snowmelt | +0.21 |
| `aspect_deg` | Topography | Slope direction, affects snowmelt | -0.21 |